In [1]:
import os
import glob
import pandas as pd
import json

# Define the exact sensitive data keys/columns to remove
csv_columns_to_remove = ['Prompt', 'Truth', 'Candidate_Left', 'Candidate_Right']
json_keys_to_remove = ['prompt', 'truth', 'candidate_left', 'candidate_right']

def scrub_json_data(data):
    """Recursively removes sensitive keys from JSON structures."""
    if isinstance(data, dict):
        return {k: scrub_json_data(v) for k, v in data.items() if k not in json_keys_to_remove}
    elif isinstance(data, list):
        return [scrub_json_data(item) for item in data]
    return data

def process_directory():
    # Find all CSV and JSONL files in the current folder, ignoring already-safe files
    csv_files = [f for f in glob.glob("*.csv") if not f.startswith("safe_")]
    jsonl_files = [f for f in glob.glob("*.jsonl") if not f.startswith("safe_")]

    print(f"Found {len(csv_files)} CSV files and {len(jsonl_files)} JSONL files to process.\n")

    # Process CSVs
    if csv_files:
        print("--- Processing CSV Files ---")
        for file in csv_files:
            print(f"Scrubbing {file}...")
            df = pd.read_csv(file)
            cols_to_drop = [c for c in csv_columns_to_remove if c in df.columns]
            df.drop(columns=cols_to_drop).to_csv(f"safe_{file}", index=False)
            print(f" -> Saved safe_{file}")
        print("\n")

    # Process JSONLs
    if jsonl_files:
        print("--- Processing JSONL Files ---")
        for file in jsonl_files:
            print(f"Scrubbing {file}...")
            with open(file, 'r', encoding='utf-8') as f_in, \
                 open(f"safe_{file}", 'w', encoding='utf-8') as f_out:
                for line in f_in:
                    if line.strip():
                        safe_data = scrub_json_data(json.loads(line))
                        f_out.write(json.dumps(safe_data) + '\n')
            print(f" -> Saved safe_{file}")

    print("\n✅ Bulk processing complete! You are now safe to share the 'safe_' files.")

if __name__ == "__main__":
    process_directory()

Found 19 CSV files and 19 JSONL files to process.

--- Processing CSV Files ---
Scrubbing DASE_Bellman_Results_qwen_8_alpha0_2.csv...
 -> Saved safe_DASE_Bellman_Results_qwen_8_alpha0_2.csv
Scrubbing DASE_Bellman_Results_GPQA_4_alpha0_2.csv...
 -> Saved safe_DASE_Bellman_Results_GPQA_4_alpha0_2.csv
Scrubbing DASE_DDM_qw_8k_15L.csv...
 -> Saved safe_DASE_DDM_qw_8k_15L.csv
Scrubbing DASE_Bellman_Results_NP_8_L15_alpha0_2.csv...
 -> Saved safe_DASE_Bellman_Results_NP_8_L15_alpha0_2.csv
Scrubbing DASE_DDM_qwll_8k_15L.csv...
 -> Saved safe_DASE_DDM_qwll_8k_15L.csv
Scrubbing IMSC70_Benchmark_8B.csv...
 -> Saved safe_IMSC70_Benchmark_8B.csv
Scrubbing DASE_DDM_Results__k8_L15_nopromptinj.csv...
 -> Saved safe_DASE_DDM_Results__k8_L15_nopromptinj.csv
Scrubbing IMSC70_Benchmark_GPQA.csv...
 -> Saved safe_IMSC70_Benchmark_GPQA.csv
Scrubbing DASE_DDM_Results__k4_nopromptinj.csv...
 -> Saved safe_DASE_DDM_Results__k4_nopromptinj.csv
Scrubbing DASE_DDM_Results_GPQA_qwll_k4_L15.csv...
 -> Saved safe_

In [3]:
import json
import re

# Define the input and output notebook filenames
input_notebook = "DASE_Analysis_corrected.ipynb"
output_notebook = "safe_DASE_Analysis.ipynb"

def update_notebook_filenames():
    try:
        # Load the notebook
        with open(input_notebook, 'r', encoding='utf-8') as f:
            notebook_data = json.load(f)
            
        # Regex pattern to find filenames ending in .csv or .jsonl
        # \b ensures we match the start of the filename
        # (?<!safe_) ensures we don't accidentally update 'safe_file.csv' to 'safe_safe_file.csv'
        pattern = re.compile(r'(?<!safe_)\b([a-zA-Z0-9_-]+\.(?:csv|jsonl))\b')
        
        updates_made = 0
        
        # Iterate through every cell in the notebook
        for cell in notebook_data.get('cells', []):
            if 'source' in cell:
                updated_source = []
                for line in cell['source']:
                    # Replace the old filename with the safe_ prefixed filename
                    new_line, count = pattern.subn(r'safe_\1', line)
                    updated_source.append(new_line)
                    updates_made += count
                cell['source'] = updated_source
                
        # Save the updated notebook
        with open(output_notebook, 'w', encoding='utf-8') as f:
            json.dump(notebook_data, f, indent=1)
            
        print(f"✅ Successfully made {updates_made} filename replacements.")
        print(f"✅ Updated notebook saved as: {output_notebook}")
        
    except FileNotFoundError:
        print(f"⚠️ Error: Could not find '{input_notebook}' in the current directory.")

if __name__ == "__main__":
    update_notebook_filenames()

✅ Successfully made 111 filename replacements.
✅ Updated notebook saved as: safe_DASE_Analysis.ipynb
